In [ ]:
import torch
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
from PIL import Image
import requests

In [ ]:
# Load the processor and model
processor = LayoutLMv3Processor.from_pretrained("nielsr/layoutlmv3-finetuned-cord")
model = LayoutLMv3ForTokenClassification.from_pretrained("nielsr/layoutlmv3-finetuned-cord")


In [ ]:
# Set the device
device = "cuda" if torch.cuda.is_available() else "mps"
model.to(device)

# Load an image
image_url = "https://example.com/path/to/your/image.jpg"
response = requests.get(image_url)
image = Image.open(BytesIO(response.content)).convert("RGB")


In [ ]:
# Preprocess the image
encoding = processor(image, return_tensors="pt")
input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)
bbox = encoding["bbox"].to(device)
pixel_values = encoding["pixel_values"].to(device)


In [ ]:
# Perform inference
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, bbox=bbox, pixel_values=pixel_values)
    logits = outputs.logits

# Get the predicted class labels
predicted_class_indices = logits.argmax(-1).squeeze().tolist()
labels = processor.tokenizer.convert_ids_to_tokens(predicted_class_indices)

# Print the results
print(labels)